# មេរៀន 03 - លំនាំរចនាសម្រាប់ភ្នាក់ងារ

នៅក្នុងមេរៀននេះ យើងស្វែងយល់ពីលំនាំរចនាដ៏មូលដ្ឋានបីសម្រាប់កសាងភ្នាក់ងារ AI ដែលមានប្រសិទ្ធភាព៖

1. **សេចក្តីណែនាំភ្នាក់ងារដែលច្បាស់** — បង្កើតបញ្ចូនដែលមានភាពច្បាស់ និងកំណត់តួនាទី ដើម្បីណែនាំអាកប្បកម្មរបស់ភ្នាក់ងារ
2. **លទ្ធផលមានរចនាសម្ព័ន្ធ ជាមួយម៉ូឌែល Pydantic** — ធានាថាភ្នាក់ងារត្រឡប់ទិន្នន័យដែលអាចទាយបាន និងបានត្រួតពិនិត្យ
3. **ភ្នាក់ងារដែលទទួលខុសត្រូវតែមួយ** — រៀបចំភ្នាក់ងារផ្តោតសំខាន់ ដែលម្នាក់ម្នាក់អាចអនុវត្តរឿងមួយឲ្យល្អ

យើងនឹងអនុវត្តលំនាំនីមួយៗទៅលើសេណារីយ៉ូសម្រាប់ **កម្មវិធីផ្ដល់អនុសាសន៍កន្លែងទេសចរណ៍**, ដោយជាការបន្តៗក្នុងការសង់ប្រព័ន្ធមួយដែលអាចផ្ដល់អនុសាសន៍កន្លែង, ពិនិត្យភាពអាចរកបាន និងដោះស្រាយការរៀបចំដំណើរការ។


## ការដំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## លំនាំទី 1: ការណែនាំភ្នាក់ងារដោយច្បាស់

លំនាំដែលមានឥទ្ធិពលបំផុតក៏មានភាពសាមញ្ញបំផុតផងដែរ៖ សរសេរការណែនាំដែលច្បាស់ និងលម្អិតសម្រាប់ភ្នាក់ងាររបស់អ្នក។

ការណែនាំល្អកំណត់:
- **នរណា** ភ្នាក់ងារជាអ្នកណា (បុគ្គលិកលក្ខណៈ និងសម្លេង)
- **អ្វី** វាគួរធ្វើអ្វី (កាតព្វកិច្ចជាដំណាក់កាល)
- **របៀប** វាគួរប្រព្រឹត្តដោយរបៀបណា (ការកំណត់ និងរចនាបថ)

ខាងក្រោម យើងបង្កើតភ្នាក់ងារជំនួយការធ្វើដំណើរមួយ ដោយមានការណែនាំច្បាស់លាស់ ដែលកំណត់ទម្រង់នៃគ្រប់ចម្លើយដែលវាផលិត។


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## លំនាំទី 2: លទ្ធផលដែលមានរចនាសម្ព័ន្ធជាមួយ Pydantic Models

អត្ថបទទ្រង់ទ្រាយសេរីមានប្រយោជន៍សម្រាប់ការពិភាក្សា ប៉ុន្តែប្រព័ន្ធដែលប្រើប្រាស់បន្តត្រូវការទិន្នន័យដែលមានរចនាសម្ព័ន្ធ។
ដោយភ្ជាប់ **Pydantic models** ជាមួយ **មុខងារ​ឧបករណ៍**, យើងអាច៖

- កំណត់ស្កីម៉ាដែលច្បាស់សម្រាប់លទ្ធផលនៃភ្នាក់ងារ
- ផ្ទៀងផ្ទាត់ចម្លើយដោយស្វ័យប្រវត្តិ
- បញ្ចូលលទ្ធផលរបស់ភ្នាក់ងារចូលទៅក្នុងលក្ខណៈដំណើរការរបស់កម្មវិធីដោយទុកចិត្តបាន

យើងក៏ណែនាំឧបករណ៍មួយដែលបញ្ជូនព័ត៌មានលម្អិតពីគោលដៅ ដើម្បីឱ្យភ្នាក់ងារអាចផ្អែកលើទិន្នន័យពិតក្នុងការផ្តល់អនុសាសន៍។


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## លំនាំ 3: ភ្នាក់ងារមានតួនាទីតែមួយ

ភារកិច្ចស្មុគស្មាញទទួលបានអត្ថប្រយោជន៍ពីការបែងចែកការងារទៅកាន់ភ្នាក់ងារជាច្រើនដែលមានការផ្តោតលើបេសកកម្មជាក់លាក់ ម្នាក់មួយមានតួនាទីតែមួយ:

- **អ្នកជំនាញគោលដៅ** ម្នាក់ ដែលស្គាល់អំពីកន្លែងនានា និងភាពអាចប្រើបាន
- **អ្នករៀបចំផែនការដឹកជញ្ជូន** ដែលទទួលខុសត្រូវចំពោះការហោះហើរ សណ្ឋាគារ និងផែនការធ្វើដំណើរ

នេះស្រដៀងនឹងគោលការណ៍វិស្វកម្មកម្មវិធីអំពី *ការបំបែកកាតព្វកិច្ច* — ភ្នាក់ងារនីមួយៗកាន់តែងាយស្រួលក្នុងការធ្វើតេស្ត ថែទាំ និងប្រសើរឡើងដោយឯករាជ្យ។


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## សេចក្តីសង្ខេប

ក្នុងមេរៀននេះ យើងបានអនុវត្តលំនាំរចនាដែលមានលក្ខណៈភ្នាក់ងារ ចំនួនបី លើសេណារីយោសម្រាប់កម្មវិធីផ្តល់អនុសាសន៍អំពីការធ្វើដំណើរ:

| លំនាំ | គំនិតសំខាន់ | អត្ថប្រយោជន៍ |
|---|---|---|
| **ការណែនាំច្បាស់លាស់** | កំណត់អត្តចរិត ភារកិច្ច និងលក្ខខណ្ឌពីដំបូង | អាកប្បកិរិយារបស់ភ្នាក់ងារមានភាពត្រឹមត្រូវ និងស្របទៅនឹងម៉ាក |
| **លទ្ធផលដែលមានរចនាសម្ព័ន្ធ** | ប្រើ Pydantic models ជាទ្រង់ទ្រាយនៃការឆ្លើយតប | លទ្ធផលដែលបានផ្ទៀងផ្ទាត់ និងអាចអានដោយម៉ាស៊ីន |
| **ការទទួលខុសត្រូវតែមួយ** | ផ្ដល់ភារកិច្ចតែមួយដែលផ្តោតសម្រាប់ភ្នាក់ងារនីមួយៗ | งាយសម្រាប់សាកល្បង ថែទាំ និងចងក្រង |

លំនាំទាំងនេះអាចចងក្រងគ្នាបានយ៉ាងធម្មជាតិ — អ្នកអាចបញ្ចូលសេចក្តីណែនាំច្បាស់លាស់ជាមួយលទ្ធផលដែលមានរចនាសម្ព័ន្ធ នៅក្នុងភ្នាក់ងារដែលមានទំនួលខុសត្រូវតែមួយ ដើម្បីស្ថាបនាប្រព័ន្ធដែលរឹងមាំ និងត្រៀមសម្រាប់ផលិតកម្ម


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំធ្វើឱ្យមានភាពត្រឹមត្រូវ សូមជ្រាបថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាដើមរបស់វាគួរត្រូវបានចាត់ទុកជាប្រភពផ្លូវការដែលអាចទុកចិត្តបាន។ សម្រាប់ព័ត៌មានសំខាន់ យើងផ្ដល់អនុសាសន៍ឱ្យប្រើការបកប្រែដោយអ្នកបកប្រែមនុស្សជំនាញ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
